# Assignment TREND

This notebook implements the four trend systems from the assignment:
- 10/30 moving average crossover
- 30/100 moving average crossover
- 80/160 moving average crossover
- 30 day breakout

I test them on the Euro, the 10 year note, and the S&P 500 through December 31, 2010.

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.4f}'.format)

## Load the data

The workbook already includes daily percent changes. I use those returns to build a clean price index starting at 100. That keeps the series continuous around futures rolls and makes the moving average and breakout rules easier to apply.

In [ ]:
def load_asset(sheet_name):
    df = pd.read_excel(
        '/Users/jlaw/projects/stern/systematic-investing/data/assignment_TREND_data.xlsx',
        sheet_name=sheet_name,
        header=1
    ).copy()

    df = df[df['#Date'].notna() & (df['#Date'] > 0)].copy()
    df['Date'] = pd.to_datetime(df['#Date'].astype(int).astype(str), format='%Y%m%d')
    df = df[df['Date'] <= pd.Timestamp('2010-12-31')].copy()

    for col in ['Open', 'High', 'Low', 'Close', 'PC(%)', 'Roll']:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['ret'] = df['PC(%)'] / 100
    df['index_close'] = 100 * (1 + df['ret']).cumprod()
    df['index_high'] = df['index_close'] * df['High'] / df['Close']
    df['index_low'] = df['index_close'] * df['Low'] / df['Close']

    return df.reset_index(drop=True)


uro = load_asset('uro')
ty = load_asset('ty')
sp = load_asset('sp')

uro[['Date', 'Close', 'PC(%)', 'index_close']].head()

## Strategy rules

For the moving average systems, I compare the short moving average with the long moving average of the 100-based price index.

For the 30 day breakout system, I compare today's adjusted close with the highest adjusted high and lowest adjusted low from the prior 30 days.

In all cases, I shift the trading signal by one day before multiplying by returns so the strategy does not use same-day information.

In [ ]:
def sharpe_ratio(returns):
    clean = returns.dropna()
    return np.sqrt(252) * clean.mean() / clean.std(ddof=1)


def max_drawdown(equity_curve, dates):
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1
    trough_index = drawdown.idxmin()
    peak_index = equity_curve.loc[:trough_index].idxmax()

    return drawdown.loc[trough_index], dates.loc[peak_index], dates.loc[trough_index]


def moving_average_strategy(df, short_window, long_window):
    result = df[['Date', 'ret', 'index_close']].copy()
    result['short_ma'] = result['index_close'].rolling(short_window).mean()
    result['long_ma'] = result['index_close'].rolling(long_window).mean()

    result['signal'] = 0.0
    result.loc[result['short_ma'] > result['long_ma'], 'signal'] = 1.0
    result.loc[result['short_ma'] < result['long_ma'], 'signal'] = -1.0
    result.loc[result[['short_ma', 'long_ma']].isna().any(axis=1), 'signal'] = np.nan

    result['strategy_ret'] = result['signal'].shift(1) * result['ret']
    result['equity'] = (1 + result['strategy_ret'].fillna(0)).cumprod()

    return result


def breakout_strategy(df, window=30):
    result = df[['Date', 'ret', 'index_close', 'index_high', 'index_low']].copy()
    result['hi30'] = result['index_high'].shift(1).rolling(window).max()
    result['low30'] = result['index_low'].shift(1).rolling(window).min()

    result['action'] = np.nan
    result.loc[result['index_close'] > result['hi30'], 'action'] = 1.0
    result.loc[result['index_close'] < result['low30'], 'action'] = -1.0

    result['position'] = result['action'].ffill().fillna(0.0)
    result.loc[result[['hi30', 'low30']].isna().any(axis=1), 'position'] = np.nan

    result['strategy_ret'] = result['position'].shift(1) * result['ret']
    result['equity'] = (1 + result['strategy_ret'].fillna(0)).cumprod()

    return result

## Part 1: Test all four systems on each instrument

In [ ]:
asset_data = {
    'URO': uro,
    'TY': ty,
    'SP': sp,
}

asset_results = {}
summary_rows = []

for asset_name, df in asset_data.items():
    asset_results[asset_name] = {}

    asset_results[asset_name]['10/30'] = moving_average_strategy(df, 10, 30)
    asset_results[asset_name]['30/100'] = moving_average_strategy(df, 30, 100)
    asset_results[asset_name]['80/160'] = moving_average_strategy(df, 80, 160)
    asset_results[asset_name]['Breakout30'] = breakout_strategy(df, 30)

    for system_name, result_df in asset_results[asset_name].items():
        clean = result_df.dropna(subset=['strategy_ret']).copy()

        summary_rows.append({
            'asset': asset_name,
            'system': system_name,
            'start_date': clean['Date'].iloc[0],
            'end_date': clean['Date'].iloc[-1],
            'avg_daily_ret': clean['strategy_ret'].mean(),
            'daily_vol': clean['strategy_ret'].std(ddof=1),
            'sharpe': sharpe_ratio(clean['strategy_ret']),
            'cum_return': (1 + clean['strategy_ret']).prod() - 1,
        })

summary = pd.DataFrame(summary_rows)
summary_display = summary.copy()

for col in ['avg_daily_ret', 'daily_vol', 'sharpe', 'cum_return']:
    summary_display[col] = summary_display[col].round(4)

summary_display

In [ ]:
sharpe_table = summary.pivot(index='asset', columns='system', values='sharpe').round(3)
sharpe_table

In [ ]:
best_systems = (
    summary.sort_values(['asset', 'sharpe'], ascending=[True, False])
    .groupby('asset')
    .first()
    .reset_index()
)

best_systems_display = best_systems[['asset', 'system', 'start_date', 'sharpe', 'cum_return']].copy()
best_systems_display['sharpe'] = best_systems_display['sharpe'].round(4)
best_systems_display['cum_return'] = best_systems_display['cum_return'].round(4)
best_systems_display

## Part 2: Equal-weight combination of the best system for each instrument

The assignment says to combine the best result for each instrument and start the portfolio when the slowest system is available. I use October 20, 1999 as the combination start date from the instructions, then keep only rows where all three selected return streams are available.

In [ ]:
combo = None

for asset_name in ['URO', 'TY', 'SP']:
    system_name = best_systems.loc[best_systems['asset'] == asset_name, 'system'].iloc[0]
    temp = asset_results[asset_name][system_name][['Date', 'strategy_ret']].copy()
    temp = temp.rename(columns={'strategy_ret': asset_name})

    if combo is None:
        combo = temp
    else:
        combo = combo.merge(temp, on='Date', how='inner')

combo = combo[combo['Date'] >= pd.Timestamp('1999-10-20')].copy()
combo['equal_weight_ret'] = combo[['URO', 'TY', 'SP']].mean(axis=1, skipna=False)
combo = combo.dropna(subset=['equal_weight_ret']).reset_index(drop=True)
combo['equal_weight_index'] = (1 + combo['equal_weight_ret']).cumprod()

equal_weight_sharpe = sharpe_ratio(combo['equal_weight_ret'])
largest_drawdown, peak_date, trough_date = max_drawdown(combo['equal_weight_index'], combo['Date'])

equal_weight_summary = pd.DataFrame({
    'portfolio': ['Equal weight'],
    'start_date': [combo['Date'].iloc[0]],
    'end_date': [combo['Date'].iloc[-1]],
    'sharpe': [equal_weight_sharpe],
    'largest_drawdown': [largest_drawdown],
    'peak_date': [peak_date],
    'trough_date': [trough_date],
})

equal_weight_summary_display = equal_weight_summary.copy()
equal_weight_summary_display['sharpe'] = equal_weight_summary_display['sharpe'].round(4)
equal_weight_summary_display['largest_drawdown'] = equal_weight_summary_display['largest_drawdown'].round(4)
equal_weight_summary_display

## Part 3: Inverse-volatility combination

Here I weight the three best system return streams in inverse proportion to trailing 20 day volatility. I shift the weights by one day before applying them so the portfolio only uses information that was known at the close of the prior day.

In [ ]:
for asset_name in ['URO', 'TY', 'SP']:
    combo['vol20_' + asset_name] = combo[asset_name].rolling(20).std()
    combo['raw_w_' + asset_name] = 1 / combo['vol20_' + asset_name]

combo['raw_weight_sum'] = combo[['raw_w_URO', 'raw_w_TY', 'raw_w_SP']].sum(axis=1)

for asset_name in ['URO', 'TY', 'SP']:
    combo['w_' + asset_name] = combo['raw_w_' + asset_name] / combo['raw_weight_sum']

combo['ivol_ret'] = (
    combo['w_URO'].shift(1) * combo['URO']
    + combo['w_TY'].shift(1) * combo['TY']
    + combo['w_SP'].shift(1) * combo['SP']
)
combo['ivol_index'] = (1 + combo['ivol_ret'].fillna(0)).cumprod()

ivol_summary = pd.DataFrame({
    'portfolio': ['Inverse vol'],
    'start_date': [combo.dropna(subset=['ivol_ret'])['Date'].iloc[0]],
    'end_date': [combo.dropna(subset=['ivol_ret'])['Date'].iloc[-1]],
    'sharpe': [sharpe_ratio(combo['ivol_ret'])],
})

ivol_summary_display = ivol_summary.copy()
ivol_summary_display['sharpe'] = ivol_summary_display['sharpe'].round(4)
ivol_summary_display

In [ ]:
print('Best system for URO:', best_systems.loc[best_systems['asset'] == 'URO', 'system'].iloc[0])
print('Best system for TY :', best_systems.loc[best_systems['asset'] == 'TY', 'system'].iloc[0])
print('Best system for SP :', best_systems.loc[best_systems['asset'] == 'SP', 'system'].iloc[0])
print()
print('Equal-weight portfolio Sharpe:', round(equal_weight_sharpe, 3))
print('Largest drawdown:', round(largest_drawdown, 4))
print('Drawdown peak date:', peak_date.date())
print('Drawdown trough date:', trough_date.date())
print('Inverse-vol portfolio Sharpe:', round(sharpe_ratio(combo['ivol_ret']), 3))